In [41]:
import os
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from spectral.io import envi

In [42]:
home = '/store/carroll/col/data/2025/rccs/subsets'

In [43]:
base='NIS01'

In [ ]:
fp_obs = glob(os.path.join(home, '*_obs.hdr'))
for fp in fp_obs:
    fid = fp.split('/')[-1].split('_obs')[0]
    print(fid)
    fp_fids = glob(os.path.join(home, f'{fid}*')) # get all of the file paths with that fid

    ymd = fid.split('_')[5]
    site = '_'.join(fid.split('_')[6:])

    obs = envi.open(fp).open_memmap()[...,9].copy() # get time from obs
    obs[obs==-9999] = np.nan
    hms = str(np.nanmean(obs)).replace('.','')[:6]
    
    new_fid = '_'.join([base, ymd, hms, site])

    for fp_old in fp_fids:
        fp_out = fp_old.replace(fid, new_fid)
        os.rename(fp_old, fp_out)

In [53]:
# go back through and fix the values that are >59 for hms minute, second
fps = glob(os.path.join(home, '*'))
for fp in fps:
    hms_old = fp.split('_')[2]
    h = hms_old[:2]
    m = hms_old[2:4]
    s = hms_old[4:]
    if int(m)>59:
        m='59'
    if int(s)>59:
        s='59'
    hms_new = h+m+s
    fp_out = fp.replace(hms_old, hms_new)
    os.rename(fp, fp_out)

In [54]:
import os
from glob import glob

fids = glob('/store/carroll/col/data/2025/rccs/subsets/*_rdn.hdr')
fids = list(set([x.split('/')[-1].split('_rdn')[0] for x in fids]))

fp_out = '/store/carroll/repos/chess-isofit/2025/3c/0_rccs/rccs_2025_fids.txt'
with open(fp_out, 'w') as f:
    f.write('\n'.join(fids))